# Testes de 100% e conciliação – imobilizado da Suzano S.A.
**Dados SINTÉTICOS** – trabalho acadêmico (Auditoria 1 – UnB, 2026.2). Preparador: matrícula 241003769.

Base normativa: NBC TA 500 – Evidência de auditoria; NBC TA 240 (teste de lançamentos).

**Antes de começar:** tenha em mãos `base_imobilizado_SIM.csv.gz` e `gabarito_erros_SIM.csv`, gerados pelo `01_gerar_base_imobilizado_SIM.ipynb`. Rode os blocos em ordem, com Shift + Enter.

## Bloco 1 – Carregar os arquivos
No Colab, pede o upload de `base_imobilizado_SIM.csv.gz` e `gabarito_erros_SIM.csv`, gerados pelo `01_gerar_base_imobilizado_SIM.ipynb`.

In [ ]:
# =====================================================================
# BLOCO 1 – Carregar os arquivos
# Testes de 100% da base SINTÉTICA do imobilizado – preparador: matrícula 241003769
# =====================================================================
import numpy as np
import pandas as pd

try:                                   # no Colab: pede o upload dos arquivos que faltarem
    from google.colab import files
    import os
    faltam = [a for a in ['base_imobilizado_SIM.csv.gz', 'gabarito_erros_SIM.csv'] if not os.path.exists(a)]
    if faltam:
        print('Envie os arquivos:', faltam)
        files.upload()
except ImportError:
    pass

LER = dict(sep=';', decimal=',', encoding='utf-8-sig')
base = pd.read_csv('base_imobilizado_SIM.csv.gz', **LER, parse_dates=['data_aquisicao', 'data_lancamento'])
gabarito = pd.read_csv('gabarito_erros_SIM.csv', **LER)
DATA_BASE = pd.Timestamp('2025-12-31')
print(f'Base: {len(base):,} bens | gabarito: {len(gabarito):,} erros')

## Bloco 2 – Conciliação com a nota 15
Compara, por classe, o custo, a depreciação acumulada e o valor contábil da base com os saldos da nota 15 da DFP 2025 (valores em R\$ mil). As diferenças são explicadas no PT 500.

In [ ]:
# =====================================================================
# BLOCO 2 – Conciliação da base com a nota 15 (R$ mil)
# =====================================================================
NOTA15 = {'Terrenos': (16_720_186, 0), 'Imóveis': (14_589_194, 4_659_310),
          'Máquinas, equipamentos e instalações': (66_353_229, 32_174_476),
          'Imobilizado em andamento': (2_764_799, 0), 'Outros': (1_830_011, 1_127_446)}
conc = base.groupby('classe')[['custo_aquisicao', 'depreciacao_acumulada', 'valor_contabil']].sum() / 1000
conc = conc.loc[list(NOTA15)]
conc['custo_nota15'] = [v[0] for v in NOTA15.values()]
conc['depr_nota15'] = [v[1] for v in NOTA15.values()]
conc['vc_nota15'] = conc['custo_nota15'] - conc['depr_nota15']
conc['dif_custo'] = (conc['custo_aquisicao'] - conc['custo_nota15']).round(2)
conc['dif_depr'] = (conc['depreciacao_acumulada'] - conc['depr_nota15']).round(2)
conc['dif_vc'] = (conc['valor_contabil'] - conc['vc_nota15']).round(2)
conc.loc['Total'] = conc.sum()
pd.set_option('display.float_format', '{:,.2f}'.format)
print(conc[['dif_custo', 'dif_depr', 'dif_vc']])

## Bloco 3 – Testes de regras em 100% dos bens
Aplica 9 testes a **todos** os 520.000 bens, sem amostragem. Cada teste procura uma situação que não deveria existir: duplicidade, depreciação maior que o custo, terreno ou obra depreciados, aquisição após a data-base, valor contábil inconsistente, vida útil fora da política e lançamentos suspeitos (fim de semana, sem segregação de funções, fora de horário ou por usuário não autorizado). Os testes T08 e T09 executam, na base simulada, o teste de lançamentos planejado no PT 240.

In [ ]:
# =====================================================================
# BLOCO 3 – Testes de regras em 100% dos bens
# =====================================================================
USUARIOS_AUTORIZADOS = {f'USR{i:03d}' for i in range(1, 26)}
hora = base['hora_lancamento'].str[:2].astype(int)
testes = {
    'T01 Duplicidade (mesma NF, fornecedor e custo)':
        base.duplicated(subset=['fornecedor_id', 'numero_nf', 'custo_aquisicao'], keep='first'),
    'T02 Depreciação acumulada maior que o custo':
        base['depreciacao_acumulada'] > base['custo_aquisicao'],
    'T03 Terreno com depreciação':
        (base['classe'] == 'Terrenos') & (base['depreciacao_acumulada'] > 0),
    'T04 Obra em andamento com depreciação':
        (base['classe'] == 'Imobilizado em andamento') & (base['depreciacao_acumulada'] > 0),
    'T05 Aquisição posterior a 31/12/2025':
        base['data_aquisicao'] > DATA_BASE,
    'T06 Valor contábil diferente de custo – depreciação':
        (base['valor_contabil'] - (base['custo_aquisicao'] - base['depreciacao_acumulada'])).abs() > 0.01,
    'T07 Vida útil fora da política (menor que 1 ou maior que 40 anos)':
        base['vida_util_anos'].notna() & ((base['vida_util_anos'] < 1) | (base['vida_util_anos'] > 40)),
    'T08 Lançamento em sábado, domingo ou aprovado pelo próprio usuário':
        (base['data_lancamento'].dt.dayofweek >= 5) | (base['usuario_aprovador'] == base['usuario_lancamento']),
    'T09 Lançamento após 20h ou por usuário não autorizado':
        (hora >= 20) | ~base['usuario_lancamento'].isin(USUARIOS_AUTORIZADOS),
}
resultado = pd.DataFrame({'bens_sinalizados': {t: int(m.sum()) for t, m in testes.items()},
                          'valor_contabil_R_mil': {t: round(base.loc[m, 'valor_contabil'].sum() / 1e3, 1) for t, m in testes.items()}})
print(resultado)

## Bloco 4 – Comparação com o gabarito
Compara os bens sinalizados por cada teste com os erros inseridos no gabarito, mostrando quantos foram encontrados e quantos falsos positivos (bens sinalizados sem erro) ocorreram.

In [ ]:
# =====================================================================
# BLOCO 4 – Comparação com o gabarito (inseridos x encontrados)
# =====================================================================
TESTE_DO_ERRO = {'E1': 'T01', 'E2': 'T02', 'E3': 'T03', 'E4': 'T04', 'E5': 'T05',
                 'E6': 'T06', 'E7': 'T07', 'E8': 'T08', 'E9': 'T09', 'E10': None}
linhas = []
for erro, cod in TESTE_DO_ERRO.items():
    ids_erro = set(gabarito.loc[gabarito['codigo_erro'] == erro, 'id_bem'])
    if cod is None:
        linhas.append([erro, len(ids_erro), '—', 0, 0, 'Não detectável por regra: exige documento (AUM, PT 530)'])
        continue
    nome = [t for t in testes if t.startswith(cod)][0]
    achados = set(base.loc[testes[nome], 'id_bem'])
    linhas.append([erro, len(ids_erro), cod, len(ids_erro & achados), len(achados - ids_erro),
                   'Todos encontrados' if ids_erro <= achados else 'Parcial'])
comp = pd.DataFrame(linhas, columns=['erro', 'inseridos', 'teste', 'encontrados', 'falsos_positivos', 'situacao'])
print(comp.to_string(index=False))

## Bloco 5 – Planilha e download
Grava `PT_500_Testes_100_SIM.xlsx` com a conciliação, o resumo dos testes, a comparação com o gabarito e uma aba com os bens sinalizados por teste.

In [ ]:
# =====================================================================
# BLOCO 5 – Gravar a planilha de exceções e baixar
# =====================================================================
cols = ['id_bem', 'classe', 'subclasse', 'unidade', 'data_aquisicao', 'vida_util_anos', 'custo_aquisicao',
        'depreciacao_acumulada', 'valor_contabil', 'fornecedor_id', 'numero_nf', 'data_lancamento',
        'hora_lancamento', 'usuario_lancamento', 'usuario_aprovador']
with pd.ExcelWriter('PT_500_Testes_100_SIM.xlsx') as w:
    pd.DataFrame({'Aviso': ['DADOS SINTÉTICOS – trabalho acadêmico UnB – preparador 241003769 – não representam registros da Suzano S.A.']}).to_excel(w, sheet_name='Leia-me', index=False)
    conc.reset_index().to_excel(w, sheet_name='Conciliação nota 15', index=False)
    resultado.reset_index(names='teste').to_excel(w, sheet_name='Resumo dos testes', index=False)
    comp.to_excel(w, sheet_name='Gabarito x achados', index=False)
    for t, m in testes.items():
        base.loc[m, cols].to_excel(w, sheet_name=t[:3], index=False)
print('Planilha PT_500_Testes_100_SIM.xlsx gravada.')
try:
    files.download('PT_500_Testes_100_SIM.xlsx')
except Exception:
    pass